# 01 — Extração dos dados do ClinicalTrials.gov

Extração dos ensaios clínicos intervencionais sobre obesidade e excesso de peso
(início entre 2010 e 2025) através da API v2 do ClinicalTrials.gov.

**Atenção:** correr este notebook descarrega uma nova versão dos dados (o ClinicalTrials.gov é atualizado diariamente).
A análise usa a extração de 12/10/2026 (8 431 estudos), guardada em `data/raw/`.

In [6]:
import requests
import time
import json
from datetime import date

## 1. Parâmetros da pesquisa
Pesquisa igual à do site (`Obesity/Overweight`, intervencionais, início entre 2010 e 2025),
para permitir validar a contagem. Só são pedidos os 16 campos necessários à análise;
os contactos pessoais dos estudos não são descarregados.

In [7]:
CAMPOS = ["NCTId", "BriefTitle", "StudyType", "StartDate", "StartDateType","InterventionType","LocationCountry","OverallStatus","WhyStopped","EnrollmentCount","PrimaryCompletionDate","PrimaryCompletionDateType","CompletionDate", "CompletionDateType","Phase","LeadSponsorClass"]


In [8]:
URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
     "query.cond": "Obesity/Overweight",
    "query.term": "AREA[StudyType]INTERVENTIONAL AND AREA[StartDate]RANGE[2010-01-01, 2025-12-31]",
    "fields": ",".join(CAMPOS),
    "pageSize": 1000,
    "countTotal": "true"
}


## 2. Extração com paginação
A API devolve no máximo 1 000 estudos por página. O ciclo pede página após página
até a resposta deixar de trazer o `nextPageToken`.


In [9]:
todos_estudos = []             
params_pagina = params.copy()   
pagina = 0

while True:                                  
    resposta = requests.get(URL, params=params_pagina, timeout=30)   
    resposta.raise_for_status()              
    dados = resposta.json()

    if pagina == 0:
        total_api = dados["totalCount"]      

    estudos_pagina = dados["studies"]
    todos_estudos.extend(estudos_pagina)      
    pagina += 1
    print(f"Página {pagina}: {len(estudos_pagina)} estudos (acumulado: {len(todos_estudos)})")

    token = dados.get("nextPageToken")        
    if token is None:
        break
    params_pagina["pageToken"] = token       

    time.sleep(0.5)                           

print(f"Recolhidos: {len(todos_estudos)} · Esperados: {total_api}")   

Página 1: 1000 estudos (acumulado: 1000)
Página 2: 1000 estudos (acumulado: 2000)
Página 3: 1000 estudos (acumulado: 3000)
Página 4: 1000 estudos (acumulado: 4000)
Página 5: 1000 estudos (acumulado: 5000)
Página 6: 1000 estudos (acumulado: 6000)
Página 7: 1000 estudos (acumulado: 7000)
Página 8: 1000 estudos (acumulado: 8000)
Página 9: 431 estudos (acumulado: 8431)
Recolhidos: 8431 · Esperados: 8431


## 3. Validação
Confirma que o número de estudos recolhidos é igual ao total indicado pela API
e que não há estudos repetidos.

In [10]:
ids = []
for estudo in todos_estudos:
    ids.append(estudo["protocolSection"]["identificationModule"]["nctId"])

print(len(ids))       
print(len(set(ids)))  

8431
8431


## 4. Guardar os dados brutos
Os estudos são guardados tal como vieram da API, em `data/raw/`, com a data de extração no nome do ficheiro.

In [11]:
caminho = f"../data/raw/estudos_obesidade_{date.today()}.json"

with open(caminho, "w", encoding="utf-8") as ficheiro:
    json.dump(todos_estudos, ficheiro, ensure_ascii=False)

print(f"Guardado em: {caminho}")

Guardado em: ../data/raw/estudos_obesidade_2026-10-12.json
